# Kaggle Diffusion LoRA Fine-Tuning

This notebook records the end-to-end Video Dataset Factory path from UCF101 videos to a Diffusers image-caption dataset and a Stable Diffusion LoRA checkpoint.

In [ ]:
!nvidia-smi

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [ ]:
%cd /kaggle/working
!rm -rf video-dataset-factory
!git clone https://github.com/karatarassul4-max/video-dataset-factory.git
%cd /kaggle/working/video-dataset-factory
!pip install -q -e '.[diffusion-finetune]'
!pip uninstall -y torchao

Repository cloned, diffusion fine-tuning extras installed, incompatible Kaggle torchao package removed.


In [ ]:
!mkdir -p scripts
!curl -L https://raw.githubusercontent.com/huggingface/diffusers/v0.37.1/examples/text_to_image/train_text_to_image_lora.py \
  -o scripts/train_text_to_image_lora.py
!grep -n 'check_min_version' scripts/train_text_to_image_lora.py | head

64:check_min_version("0.37.0")


In [ ]:
from pathlib import Path
import shutil

root = Path('/kaggle/input/datasets/matthewjansen/ucf101-action-recognition')
files = sorted(root.rglob('*.avi'))[:100]
Path('data/ucf_sample').mkdir(parents=True, exist_ok=True)
for idx, src in enumerate(files, 1):
    shutil.copy2(src, Path('data/ucf_sample') / f'ucf_{idx:03d}.avi')
print('found:', len(files))
print('copied:', len(list(Path('data/ucf_sample').glob('*.avi'))))

found: 100
copied: 100


In [ ]:
!vdf process-folder data/ucf_sample --output outputs/manifest.jsonl
!vdf dedupe-manifest outputs/manifest.jsonl --output outputs/manifest_deduped.jsonl

Manifest Dedupe
records: 100
duplicates: 8
output: outputs/manifest_deduped.jsonl


In [ ]:
import json
from pathlib import Path
from collections import Counter

rows = [json.loads(line) for line in Path('outputs/manifest_deduped.jsonl').read_text().splitlines() if line.strip()]
kept = [r for r in rows if r.get('keep', True)]
rejected = [r for r in rows if not r.get('keep', True)]
print('total:', len(rows))
print('kept:', len(kept))
print('rejected:', len(rejected))
print(Counter(reason for r in rejected for reason in r.get('reject_reasons', [])))

total: 100
kept: 0
rejected: 100
Counter({'resolution_too_low': 100, 'text_or_watermark_likely': 48, 'too_static': 12, 'near_duplicate': 8, 'duration_too_long': 1})


In [ ]:
# Create a relaxed manifest for the LoRA smoke test while preserving original reject reasons.
import json
from pathlib import Path

src = Path('outputs/manifest_deduped.jsonl')
dst = Path('outputs/manifest_lora_relaxed.jsonl')
rows = [json.loads(line) for line in src.read_text().splitlines() if line.strip()]
relaxed = []
for r in rows:
    r = dict(r)
    r['original_keep'] = r.get('keep')
    r['original_reject_reasons'] = r.get('reject_reasons', [])
    r['keep'] = True
    r['reject_reasons'] = []
    relaxed.append(r)
dst.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in relaxed) + '\n', encoding='utf-8')
print('clips:', len(relaxed))

clips: 100


In [ ]:
!vdf prepare-diffusion-lora-data outputs/manifest_lora_relaxed.jsonl \
  --output-dir outputs/diffusion_lora_dataset \
  --frames-per-clip 1 \
  --max-clips 100 \
  --resolution 512
!wc -l outputs/diffusion_lora_dataset/metadata.jsonl

Diffusion LoRA Dataset
source clips: 100
images: 100
skipped: 0
100 outputs/diffusion_lora_dataset/metadata.jsonl


In [ ]:
!accelerate launch --num_processes=1 --num_machines=1 --mixed_precision=fp16 --dynamo_backend=no scripts/train_text_to_image_lora.py \
  --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
  --train_data_dir=outputs/diffusion_lora_dataset \
  --resolution=512 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=120 \
  --learning_rate=1e-4 \
  --lr_scheduler=constant \
  --lr_warmup_steps=0 \
  --rank=8 \
  --mixed_precision=fp16 \
  --seed=13 \
  --output_dir=outputs/diffusion_lora

***** Running training *****
Num examples = 100
Num Epochs = 5
Instantaneous batch size per device = 1
Total train batch size (w. parallel, distributed & accumulation) = 4
Gradient Accumulation steps = 4
Total optimization steps = 120
Steps: 100%|█████| 120/120 [03:13<00:00, 1.61s/it, lr=0.0001, step_loss=0.0495]
Model weights saved in outputs/diffusion_lora/pytorch_lora_weights.safetensors


In [ ]:
!find outputs/diffusion_lora -maxdepth 2 -type f | sort
!ls -lh outputs/diffusion_lora/pytorch_lora_weights.safetensors

outputs/diffusion_lora/pytorch_lora_weights.safetensors
-rw-r--r-- 1 root root 6.2M Aug 21 09:40 outputs/diffusion_lora/pytorch_lora_weights.safetensors


## Result

The run completed a Stable Diffusion LoRA fine-tune from a Video Dataset Factory manifest-derived Diffusers dataset. The production quality filters rejected UCF101 for low resolution, so a relaxed manifest was used only to validate the end-to-end fine-tuning path while preserving the original reject reasons for auditability.